# DL01 — ResNet: Deep Residual Learning for Image Recognition


## 0. Paper Information
| | |
|---|---|
| **Paper** | Deep Residual Learning for Image Recognition |
| **Authors** | He, K., Zhang, X., Ren, S., & Sun, J. |
| **Venue** | CVPR 2016 |
| **arXiv** | [1512.03385](https://arxiv.org/abs/1512.03385) |
| **Key Contribution** | Residual connections solve degradation in very deep networks |
| **This notebook** | Full ResNet implementation + CIFAR-10 training |

**Core Idea:** Instead of learning H(x) directly, learn the residual F(x) = H(x) - x.  
The shortcut connection adds x back: output = F(x) + x  
This makes optimization dramatically easier for deep networks (50–152+ layers).


---
## 1. Architecture & Key Design Decisions
### 1.1 Why Residual Connections?
- Plain deep networks suffer from **degradation**: adding more layers hurts training accuracy
- This is NOT caused by overfitting — it's an optimization problem
- Residual connections provide a **gradient highway**: gradients can flow directly through shortcuts

### 1.2 Two Block Types
| Block | Used in | Layers per block |
|-------|---------|------------------|
| BasicBlock | ResNet-18, ResNet-34 | 2 × (3×3 conv) |
| Bottleneck | ResNet-50, ResNet-101, ResNet-152 | 1×1 → 3×3 → 1×1 conv |

### 1.3 Architecture Summary (ResNet-18)
```
Input (3×224×224)
  → Conv1 (7×7, 64, stride=2) + BN + ReLU
  → MaxPool (3×3, stride=2)
  → Layer1: 2× BasicBlock(64)
  → Layer2: 2× BasicBlock(128, stride=2)
  → Layer3: 2× BasicBlock(256, stride=2)
  → Layer4: 2× BasicBlock(512, stride=2)
  → GlobalAvgPool → FC(num_classes)
```


---
## 2. Mathematical Foundation
### 2.1 Residual Mapping (Eq. 1 in paper)
Standard layer: $y = \mathcal{H}(x)$

Residual reformulation: $y = \mathcal{F}(x, \{W_i\}) + x$

where $\mathcal{F}$ is the residual to be learned.

### 2.2 Identity Shortcut vs. Projection Shortcut
When input/output dimensions match: $y = \mathcal{F}(x) + x$ (identity shortcut, zero extra params)

When dimensions differ: $y = \mathcal{F}(x) + W_s x$ (learned projection, Eq. 2)

### 2.3 Batch Normalization
Applied after each conv, before ReLU: $\hat{x} = \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$, then scale & shift.


---
## 3. Core Implementation


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


### 3.1 Basic Block (for ResNet-18/34)


In [ ]:
class BasicBlock(nn.Module):
    """Two 3x3 convolutions with residual shortcut.
    Used in ResNet-18 and ResNet-34 (He et al., 2016, Fig. 2 left).
    """
    expansion = 1  # output channels = planes * expansion

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(planes)

        # Projection shortcut when dimensions change (Option B in paper)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes * self.expansion, 1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * self.expansion)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)  # residual addition
        return F.relu(out)


### 3.2 Bottleneck Block (for ResNet-50/101/152)


In [ ]:
class Bottleneck(nn.Module):
    """1x1 → 3x3 → 1x1 conv with 4x channel expansion.
    Used in ResNet-50+ (He et al., 2016, Fig. 2 right).
    """
    expansion = 4

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 1, bias=False)
        self.bn1   = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, planes * self.expansion, 1, bias=False)
        self.bn3   = nn.BatchNorm2d(planes * self.expansion)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes * self.expansion, 1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * self.expansion)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out += self.shortcut(x)
        return F.relu(out)


### 3.3 Full ResNet


In [ ]:
class ResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=1000, small_input=False):
        """General ResNet constructor.

        Args:
            block: BasicBlock or Bottleneck
            num_blocks: list of 4 ints, blocks per stage
            num_classes: output classes (1000 for ImageNet, 10 for CIFAR-10)
            small_input: if True, adapt stem for 32x32 inputs (CIFAR-10)
        """
        super().__init__()
        self.in_planes = 64

        if small_input:  # CIFAR-10: 32x32
            self.stem = nn.Sequential(
                nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False),
                nn.BatchNorm2d(64), nn.ReLU(inplace=True)
            )
        else:            # ImageNet: 224x224
            self.stem = nn.Sequential(
                nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False),
                nn.BatchNorm2d(64), nn.ReLU(inplace=True),
                nn.MaxPool2d(3, stride=2, padding=1)
            )

        self.layer1 = self._make_layer(block, 64,  num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        layers = [block(self.in_planes, planes, stride)]
        self.in_planes = planes * block.expansion
        for _ in range(1, num_blocks):
            layers.append(block(self.in_planes, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x)
        x = self.layer3(x); x = self.layer4(x)
        x = self.avgpool(x)
        return self.fc(x.flatten(1))


def resnet18(num_classes=1000, small_input=False):
    return ResNet(BasicBlock, [2,2,2,2], num_classes, small_input)

def resnet34(num_classes=1000, small_input=False):
    return ResNet(BasicBlock, [3,4,6,3], num_classes, small_input)

def resnet50(num_classes=1000, small_input=False):
    return ResNet(Bottleneck, [3,4,6,3], num_classes, small_input)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

for name, fn in [('ResNet-18', resnet18), ('ResNet-34', resnet34), ('ResNet-50', resnet50)]:
    m = fn(num_classes=1000)
    print(f'{name}: {count_params(m)/1e6:.1f}M parameters')


---
## 4. Training Pipeline


In [ ]:
# ── Data: CIFAR-10 (32x32) ───────────────────────────────────────────────
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

train_set = datasets.CIFAR10(root='./data', train=True,  download=True, transform=transform_train)
test_set  = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False, num_workers=2)
print(f'Train: {len(train_set)}, Test: {len(test_set)}')


In [ ]:
# ── Model, optimizer, scheduler ──────────────────────────────────────────
model = resnet18(num_classes=10, small_input=True).to(device)
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
criterion = nn.CrossEntropyLoss()


def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
        correct += (out.argmax(1) == y).sum().item()
        total += len(y)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        correct += (model(X).argmax(1) == y).sum().item()
        total += len(y)
    return correct / total


# Training loop (set EPOCHS=20 for real training; 3 for quick demo)
EPOCHS = 3  # change to 20 for full training
history = {'train_loss': [], 'train_acc': [], 'test_acc': []}

for epoch in range(1, EPOCHS + 1):
    loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
    te_acc = evaluate(model, test_loader)
    scheduler.step()
    history['train_loss'].append(loss)
    history['train_acc'].append(tr_acc)
    history['test_acc'].append(te_acc)
    print(f'Epoch {epoch:02d} | Loss {loss:.4f} | Train {tr_acc:.3f} | Test {te_acc:.3f}')


---
## 5. Results & Evaluation


In [ ]:
# Paper benchmark: ResNet-18 on CIFAR-10 → ~93% test accuracy
# (with 20 epochs + cosine annealing)
best_test = max(history['test_acc'])
print(f'Best test accuracy: {best_test:.4f} ({best_test*100:.2f}%)')
print(f'Paper baseline (ResNet-18, CIFAR-10): ~93.0%')


---
## 6. Visualization


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(history['train_loss'], 'b-o', label='Train Loss')
ax1.set_title('Training Loss'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot([a*100 for a in history['train_acc']], 'b-o', label='Train Acc')
ax2.plot([a*100 for a in history['test_acc']],  'r-s', label='Test Acc')
ax2.axhline(93, color='gray', linestyle='--', label='Paper baseline ~93%')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('%'); ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('ResNet-18 on CIFAR-10', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
